In [1]:
import pandas as pd
from tqdm import tqdm
import os

from m_features import add_all_m_features


Using device: mps


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [2]:
import gc
import torch

In [9]:
INPUT_FILE = "text_dict.jsonl"     
OUTPUT_DIR = "message_features"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 50000  # safer for large transformer models
BATCH_SIZE = 64      # increase if GPU


In [10]:
total_lines = sum(1 for _ in open(INPUT_FILE))
total_chunks = (total_lines // CHUNK_SIZE) + 1

print(f"Total posts: {total_lines}")
print(f"Total chunks: {total_chunks}")


Total posts: 1987001
Total chunks: 40


In [8]:
reader = pd.read_json(INPUT_FILE, lines=True, chunksize=CHUNK_SIZE)

for i, chunk in tqdm(enumerate(reader), total=total_chunks, desc="Chunks"):

    # Ensure required columns exist
    if "text" not in chunk.columns:
        raise ValueError("JSONL must contain a 'text' column")

    # Apply full feature pipeline
    df_features = add_all_m_features(
        chunk,
        batch_size=BATCH_SIZE
    )

    # Save chunk
    output_path = os.path.join(OUTPUT_DIR, f"features_chunk_{i}.parquet")
    df_features.to_parquet(output_path, index=False)

    del df_features
    gc.collect()
    torch.mps.empty_cache()


print("\nAll chunks processed successfully.")


Chunks:   0%|          | 1/398 [02:47<18:27:49, 167.43s/it]


KeyboardInterrupt: 

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("message_features/features_chunk_*.parquet"))

for f in files[:1]:
    df_chunk = pd.read_parquet(f)

    # Do analysis here
    print(df_chunk)
    del df_chunk


                                              post_uri  \
0    at://did:plc:6nus4szvjo4sb2ntctedjbtd/app.bsky...   
1    at://did:plc:6nus4szvjo4sb2ntctedjbtd/app.bsky...   
2    at://did:plc:6nus4szvjo4sb2ntctedjbtd/app.bsky...   
3    at://did:plc:6nus4szvjo4sb2ntctedjbtd/app.bsky...   
4    at://did:plc:6nus4szvjo4sb2ntctedjbtd/app.bsky...   
..                                                 ...   
495  at://did:plc:mo5olzazxi4zd3meb2bsclhh/app.bsky...   
496  at://did:plc:wqj7pon6wnd6kof66pdqeony/app.bsky...   
497  at://did:plc:kjsl6b5t5klf5cbtnkyxq3ar/app.bsky...   
498  at://did:plc:udaj4eodg5snflwbpwlik4ql/app.bsky...   
499  at://did:plc:3nka6y347mrgyt6nky2uu2kp/app.bsky...   

                                                  text  text_len  word_count  \
0    Oh, the unintended consequences of consolidati...       294          53   
1    Today was a slog, but I learned a bunch, and f...       219          39   
2    Out with the castoff, broken chair, and back t...       29